# 8. Building a dataset from peak files

The classification dataset used in the fine-tuning notebook did not arrive as a finished
label matrix. It was built from ENCODE IDR narrowPeak files, and this notebook rebuilds it
so you can do the same for an RBP of your own.

The tiles and their sequences already exist; what gets derived here is the **label** —
does this 600 nt tile overlap a peak? That is the part you would actually have to produce
for a new experiment.

At the end we check the rebuilt labels against the ones that shipped. They should match
exactly, and if they do not, the notebook says so rather than carrying on.

In [1]:
import gzip
import pathlib
from collections import defaultdict

import numpy as np
import torch

import parnet_tutorials as pt

peaks_dir = pathlib.Path(pt.load_config()['resources']['tutorial_peaks'])
shipped = torch.load(pt.classification_path('peaks_train'), map_location='cpu', weights_only=False)

tracks = shipped['rbps']
print(f'{len(shipped["names"]):,} tiles of {shipped["seq"].shape[1]} nt')
print(f'{len(tracks)} tracks')
print('peak files:', len(sorted(peaks_dir.glob('*.bed.gz'))))

94,743 tiles of 600 nt
12 tracks
peak files: 12


## What a narrowPeak file looks like

Ten columns. We need four of them: the chromosome, the start, the end and the strand.
Columns 7 to 10 hold the signal, p-value, q-value and summit offset, which a stricter
pipeline would threshold on; here the file is already the IDR-reproducible set, so a peak
being present is the filter.

In [2]:
example = peaks_dir / 'QKI_HepG2.idr.tutorial-chroms.bed.gz'
with gzip.open(example, 'rt') as handle:
    for _ in range(3):
        fields = handle.readline().rstrip('\n').split('\t')
        print(fields)

print()
print('columns we use: 0 chrom, 1 start, 2 end, 5 strand')

['chr2', '241250904', '241250940', 'QKI_HepG2_IDR', '1000', '-', '3.33584348436297', '3.95340378518435', '-1', '-1']
['chr8', '118027137', '118027182', 'QKI_HepG2_IDR', '1000', '-', '4.32260737868237', '3.74848174411885', '-1', '-1']
['chr2', '158492773', '158492841', 'QKI_HepG2_IDR', '1000', '+', '4.61731158155855', '7.58956629024155', '-1', '-1']

columns we use: 0 chrom, 1 start, 2 end, 5 strand


## Tile names carry their own coordinates

Each tile is named `chrom:start-end:strand`, so labelling is an interval overlap between
the tile and the peaks of one track. No genome FASTA is needed.

In [3]:
print('first three tiles:', shipped['names'][:3])

chroms = sorted({n.split(':')[0] for n in shipped['names']})
print('chromosomes in the training split:', chroms)
print()
print('train chr1+chr10 | validation chr2+chr9 | test chr3+chr8')
print('Splitting by chromosome keeps a tile and its homologous neighbours on the same side,')
print('which a random split would not: adjacent tiles overlap in sequence.')

first three tiles: ['chr1:8025553-8026153:-', 'chr1:100903275-100903875:+', 'chr10:112725133-112725733:+']
chromosomes in the training split: ['chr1', 'chr10']

train chr1+chr10 | validation chr2+chr9 | test chr3+chr8
Splitting by chromosome keeps a tile and its homologous neighbours on the same side,
which a random split would not: adjacent tiles overlap in sequence.


## Labelling

Two decisions worth stating, because both change the labels:

**Strand matters.** eCLIP is a stranded assay and the tiles carry a strand, so a peak on
the opposite strand is not evidence of binding for that tile. Ignoring strand would roughly
double the apparent positive rate with noise.

**Any overlap counts.** A tile is positive if a peak overlaps it by at least one
nucleotide. A stricter rule — requiring the peak summit, or a minimum overlap — is equally
defensible and would give a smaller, cleaner positive set.

In [4]:
def load_peaks(path):
    """Peak starts and ends per (chromosome, strand), sorted by start."""
    buckets = defaultdict(list)
    with gzip.open(path, 'rt') as handle:
        for line in handle:
            f = line.rstrip('\n').split('\t')
            buckets[(f[0], f[5])].append((int(f[1]), int(f[2])))
    return {
        key: (np.array([s for s, _ in sorted(spans)]), np.array([e for _, e in sorted(spans)]))
        for key, spans in buckets.items()
    }


def label_tiles(names, peaks):
    """True where a tile overlaps at least one peak on the same strand."""
    labels = np.zeros(len(names), dtype=bool)
    for i, name in enumerate(names):
        chrom, span, strand = name.split(':')
        start, end = (int(x) for x in span.split('-'))
        if (chrom, strand) not in peaks:
            continue
        peak_starts, peak_ends = peaks[(chrom, strand)]
        # A peak overlaps when it begins before the tile ends and ends after the tile
        # begins. Searching the sorted starts bounds the candidates to check.
        hi = np.searchsorted(peak_starts, end, side='left')
        labels[i] = bool(hi and (peak_ends[:hi] > start).any())
    return labels


rebuilt = np.zeros((len(shipped['names']), len(tracks)), dtype=bool)
for j, track in enumerate(tracks):
    peaks = load_peaks(peaks_dir / f'{track}.idr.tutorial-chroms.bed.gz')
    rebuilt[:, j] = label_tiles(shipped['names'], peaks)
    print(f'{track:16s} {rebuilt[:, j].sum():5d} positive tiles  '
          f'({rebuilt[:, j].mean():.4f} of the split)')

AGGF1_HepG2        164 positive tiles  (0.0017 of the split)


AGGF1_K562         465 positive tiles  (0.0049 of the split)


HNRNPK_HepG2       587 positive tiles  (0.0062 of the split)


HNRNPK_K562        358 positive tiles  (0.0038 of the split)


PTBP1_HepG2       1134 positive tiles  (0.0120 of the split)


...

QKI_K562           424 positive tiles  (0.0045 of the split)


RBFOX2_HepG2       651 positive tiles  (0.0069 of the split)


RBFOX2_K562        432 positive tiles  (0.0046 of the split)


TRA2A_HepG2        145 positive tiles  (0.0015 of the split)


TRA2A_K562         273 positive tiles  (0.0029 of the split)


## Check against what shipped

A dataset build that is not checked is a dataset you hope is right.

In [5]:
expected = shipped['labels'].numpy()
disagreements = int((rebuilt != expected).sum())

print(f'cells compared : {rebuilt.size:,}')
print(f'disagreements  : {disagreements}')
assert disagreements == 0, 'the rebuilt labels differ from the ones that shipped'
print()
print('Identical. The shipped labels are reproducible from the peak files in this bundle.')

cells compared : 1,136,916
disagreements  : 0

Identical. The shipped labels are reproducible from the peak files in this bundle.


## Saving your own

The same three keys are all a dataset needs: the sequences, the labels, and the tile names
that let you trace any row back to the genome. `rbps` records the column order, without
which the label matrix is unreadable.

In [6]:
out = {
    'seq': shipped['seq'],
    'labels': torch.from_numpy(rebuilt),
    'names': shipped['names'],
    'rbps': tracks,
}

destination = pathlib.Path('my_dataset.pt')
torch.save(out, destination)
reloaded = torch.load(destination, map_location='cpu', weights_only=False)

print('keys :', sorted(reloaded))
print('seq  :', tuple(reloaded['seq'].shape), reloaded['seq'].dtype)
print('label:', tuple(reloaded['labels'].shape), reloaded['labels'].dtype)
print('size :', f'{destination.stat().st_size / 1e6:.1f} MB')
destination.unlink()

keys : ['labels', 'names', 'rbps', 'seq']
seq  : (94743, 600) torch.uint8
label: (94743, 12) torch.bool
size : 61.3 MB


## What to take away

Positive rates here run from 0.15 to 1.4 per cent. That is the single most important
property of this dataset and it governs how you evaluate: a model that answers "no" to
every tile scores about 99 per cent accuracy. Report AUROC and AUPRC, and read AUPRC
against the track's own positive rate rather than against 0.5.

Scaling this up is mostly bookkeeping rather than new logic. Whole-genome peak files and
the full tile set use the same two functions; what grows is the time spent reading, which
is why the tutorial chromosomes are chr1+chr10, chr2+chr9 and chr3+chr8 rather than
everything.